# Project Documentation: Credit Risk Modeling & Portfolio Optimization
**End-to-End Machine Learning Pipeline, Experiment Tracking, Model Calibration, and Business Policy Optimization**

---

## 1. Executive Summary & Business Objective
In consumer credit lending, financial institutions must continuously balance two opposing objectives:
* **Portfolio Growth:** Maximizing loan approvals to increase loan volume and generate interest revenue.
* **Risk Mitigation:** Controlling default rates to minimize capital write-offs.

Setting an approval cutoff too high strangles growth and profitability by declining creditworthy applicants. Setting it too low allows high-risk borrowers to accumulate severe default losses.

This project builds a complete, reproducible machine learning system using the Kaggle **Loan Default Prediction Challenge** dataset. The pipeline covers:
1. Exploratory data analysis and statistical feature engineering (Hypothesis Testing, Weight of Evidence, Information Value).
2. Model training across linear and gradient-boosted architectures (Logistic Regression, LightGBM, CatBoost).
3. Experiment tracking and model artifact logging using MLflow.
4. Probability calibration analysis using Brier scores and reliability curves.
5. Financial decision optimization connecting calibrated default probabilities to net portfolio profits.

---

## 2. Data Architecture & Dataset Profiling
The pipeline ingests raw historical loan performance records from Kaggle (`nikhil1e9/loan-default`).

### 2.1 Dataset Schema & Missing Values
* **Sample Size:** 255,347 loan records across 18 columns.
* **Missing Values:** Zero null values across the entire dataset; no missing-value imputation is required.
* **Feature Breakdown:**
  * **Identifier:** `LoanID`.
  * **Continuous Financial Features:** `Age`, `Income`, `LoanAmount`, `CreditScore`, `MonthsEmployed`, `NumCreditLines`, `InterestRate`, `LoanTerm`, and `DTIRatio`.
  * **7 Categorical Features:** `Education`, `EmploymentType`, `MaritalStatus`, `HasMortgage`, `HasDependents`, `LoanPurpose`, and `HasCoSigner`.
  * **Target Variable:** `Default` (binary indicator: 1 = defaulted, 0 = fully repaid).

### 2.2 Class Imbalance
The population default rate is **11.6%** (roughly 1 default for every 8 non-defaults). Because the target is imbalanced, accuracy is an inadequate metric. The evaluation framework emphasizes:
* **ROC-AUC & Gini Coefficient ($2 \times \text{AUC} - 1$):** Overall ranking discrimination.
* **Precision-Recall AUC & F1-Score:** Minority-class identification performance.
* **Brier Score:** Accuracy of predicted probability scores.

---

## 3. Statistical Feature Engineering
Rather than relying on one-hot encoding (which creates sparse matrices and hides non-linear multi-variable interactions), the pipeline uses statistical encoding based on credit industry best practices.

### 3.1 Hypothesis Testing: Individual Categories
We test whether the observed default rate for each standalone category deviates significantly from the 11.6% population baseline using two-sided **Proportion Z-Tests**:
* **Null Hypothesis ($H_0$):** The category default rate equals the baseline population rate (11.6%).
* **Alternative Hypothesis ($H_1$):** The category default rate differs significantly from the baseline ($p < 0.05$).

### 3.2 Hypothesis Testing: Stratified Borrower Profiles
Borrower creditworthiness is determined by combinations of traits (e.g., employment stability combined with marital status and debt obligations). We create a composite feature (`stratify_col`) that cross-tabulates all 7 categorical attributes. Proportion testing confirms that these combined strata reveal distinct risk subgroups that diverge much further from the population baseline than individual categories alone.

### 3.3 Information Value (IV) & Weight of Evidence (WoE)
To measure predictive power, we compute the **Weight of Evidence (WoE)** and **Information Value (IV)**:

$$\text{WoE} = \ln\left(\frac{\% \text{ Non-Defaults}}{\% \text{ Defaults}}\right)$$

$$IV = \sum \left( \% \text{ Non-Defaults} - \% \text{ Defaults} \right) \times \text{WoE}$$

* **Individual Features:** `EmploymentType` is the only standalone feature with meaningful predictive power ($\text{IV} > 0.20$). The remaining 6 features exhibit weak standalone predictive power ($\text{IV} < 0.02$).
* **Stratified Feature:** The composite feature (`stratify_col`) achieves an Information Value of **0.14** (medium predictive power tier). Combining features successfully recovers interaction signals that isolated features miss.

### 3.4 Leverage Engineering: Loan-to-Income (LTI) Ratio
We engineer the **Loan-to-Income (LTI) Ratio** as a direct proxy for debt leverage:

$$\text{LTI Ratio} = \frac{\text{Loan Amount}}{\text{Annual Income}}$$

This standardizes borrowing scale relative to annual income, providing decision trees with an intuitive split feature.

### 3.5 Production Transformation Pipeline (`feature_engineering`)
To prevent data leakage, all statistical transformations (WoE mappings and target probabilities) are calculated **strictly on the training split**. These mappings are stored in Python dictionary lookup tables and applied to Validation, Test, and real-time production records via a modular `feature_engineering()` function.

---

## 4. Model Architectures & Training Pipelines
We evaluate three distinct machine learning paradigms:

### 4.1 Logistic Regression Pipeline
* **Rationale:** Because the upfront feature engineering creates linear WoE and probability mappings, categorical features share a linear relationship with the log-odds of default.
* **Architecture:** Scikit-Learn `Pipeline` with `StandardScaler` and `LogisticRegression(penalty='l2', solver='saga', class_weight='balanced')`.
* **Strength:** Exceptional interpretability, fast training, and naturally calibrated probability outputs.

### 4.2 LightGBM (Gradient Boosted Trees)
* **Rationale:** Gradient boosted decision trees capture complex interactions natively through leaf-wise tree growth.
* **Architecture:** `LGBMClassifier` paired with `ColumnTransformer` (standard scaling on continuous features and target encoding on categorical features).
* **Strength:** Strong tabular classification performance and native support for class imbalance via `scale_pos_weight=8.5`.

### 4.3 CatBoost (Categorical Boosting)
* **Rationale:** CatBoost utilizes symmetric (oblivious) trees and ordered target encoding to resist overfitting and prediction shift.
* **Architecture:** `CatBoostClassifier` with native categorical feature indexing (`cat_features`) and `auto_class_weights='Balanced'`.
* **Strength:** High stability and robust performance directly on categorical indices.

---

## 5. Validation Strategy & MLflow Tracking

### 5.1 Stratified Partitioning Strategy
To ensure realistic evaluation and prevent data leakage:
* **Test Set (20%):** Blind holdout dataset for final model evaluation and policy simulation.
* **Validation Set (16% of total):** Used for early stopping, hyperparameter tuning, and probability calibration.
* **Training Set (64% of total):** Used to compute feature mappings and train model weights.

*Stratification on both the default label and composite category guarantees identical class balance and borrower distributions across all three splits.*

### 5.2 Centralized Experiment Tracking with MLflow
All model runs are tracked through a standardized `MLflowLogger` wrapper:
* **Metrics Tracked:** ROC-AUC, Gini, Precision, Recall, F1-Score, and Brier Score on both validation and test sets.
* **Artifacts Stored:** Confusion matrices, ROC curves, Precision-Recall curves, and model weights.
* **Local Inspection:** The MLflow UI is launched via:
  ```bash
  python -m mlflow ui --host 127.0.0.1 --port 5000
  ```
  Accessible at **http://127.0.0.1:5000**.

---

## 6. Model Calibration & Probability Reliability

### 6.1 Why Calibration Matters in Credit Underwriting
In credit underwriting, classification rankings (AUC) are insufficient. The model's outputs must represent **true empirical probabilities**:
* If 1,000 applicants are assigned a 20% probability of default, exactly 200 should default.
* Uncalibrated probabilities lead to flawed expected loss calculations and inaccurate risk-based loan pricing.

### 6.2 Diagnostic Metrics: Brier Score & Reliability Diagrams
We evaluate calibration using:
1. **The Brier Score (Mean Squared Error of probabilities):**
   $$\text{Brier Score} = \frac{1}{N} \sum_{i=1}^{N} (p_i - y_i)^2$$
2. **Reliability Diagrams (Calibration Curves):** Plotting predicted probability bins against observed default rates. Perfect calibration follows the diagonal line ($y = x$).

### 6.3 Post-Hoc Calibration via `CalibratedClassifierCV`
We apply `CalibratedClassifierCV` (Isotonic Regression and Platt Scaling) to evaluate and correct probability distortions.

### 6.4 Key Finding: Linear WoE Model vs. Tree Models
* **CatBoost:** The uncalibrated baseline exhibited an S-curve distortion and a Brier Score of **~0.20** due to iterative gradient boosting with class weights. Applying Isotonic Calibration aligned the reliability curve closely to the diagonal line.
* **Logistic Regression:** The baseline achieved an outstanding Brier Score of **0.091** out of the box. Post-hoc calibration yielded no significant change ($pprox 0.00006$ difference). Linear models trained with WoE and target probabilities are inherently well-calibrated.

---

## 7. Financial Optimization: Portfolio Growth vs. Default Losses

### 7.1 Decision Threshold Scanning
The standard 0.5 classification threshold ignores the asymmetric costs of credit lending. A missed default (false negative) loses the loan principal, whereas a rejected good borrower (false positive) merely forfeits the expected interest.

We scanned decision thresholds from $0.00$ to $1.00$ to map out precision, recall, and financial trade-offs.

### 7.2 Total Predicted Interest Optimization Function
To determine the optimal underwriting rule, we model the portfolio financial objective function across decision thresholds ($t$):

$$\text{Net Profit}(t) = \sum_{\text{Approved Non-Defaulters}} \text{Interest Earned} - \sum_{\text{Approved Defaulters}} \text{Capital Loss}$$

Where:
* A borrower is approved if predicted default probability $p \le t$.
* $\text{Interest Earned} = \text{Loan Amount} \times \left[(1 + r)^{\text{Term}} - 1\right]$.
* $\text{Capital Loss} \approx \text{Loan Amount}$.

### 7.3 Profit-Maximizing Threshold Selection ($t = 0.35$)
Simulating net return across the threshold continuum demonstrates that the Total Predicted Interest curve reaches its maximum at **$t = 0.35$**:
* **Below 0.35:** Overly conservative; approves too few loans and sacrifices profitable interest revenue.
* **Above 0.35:** Overly permissive; default losses quickly exceed marginal interest revenue.
* **At 0.35:** Captures maximum net portfolio profit while filtering out high-risk applicants.

---

## 8. Operational Underwriting Recommendations & Production Deployment

### 8.1 Recommended Underwriting Policy
Based on the optimization results, we recommend a three-tiered underwriting decision framework:

| Risk Tier | Default Probability ($p$) | Underwriting Decision | Pricing & Policy Terms |
| :--- | :--- | :--- | :--- |
| **Tier 1 (Prime)** | $p \le 0.20$ | **Instant Automated Approval** | Best available market interest rate; standard documentation. |
| **Tier 2 (Near-Prime)** | $0.20 < p \le 0.35$ | **Conditional Approval** | Risk-based interest rate premium; requires secondary verification or co-signer. |
| **Tier 3 (Subprime / Decline)** | $p > 0.35$ | **Automated Decline** | Decline application to protect institutional capital; offer credit repair pathways. |

### 8.2 Production Deployment Architecture
1. **Feature Pipeline Serialization:** Store WoE and probability lookup dictionaries as lightweight JSON files for instant $O(1)$ feature encoding in real-time scoring APIs.
2. **Model Serving:** Serve the calibrated model via a REST API endpoint; output both the continuous probability score and the discrete underwriting tier.
3. **Continuous Monitoring:** Track population stability index (PSI) and monitoring dashboards to detect feature drift and maintain calibration over economic cycles.
